# Mode prediction

### Import Modules

In [2]:
import h5py
import numpy as np
from collections import Counter
from sklearn import svm
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import joblib
from sklearn.metrics import accuracy_score, auc, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score, cross_val_predict, GridSearchCV

### reading the principal components

In [3]:
with h5py.File("../visualisations/MODE_PRINCIPAL_COMPS.h5", "r") as f:
    X = f['features'][:]
    y = f['labels'][:]

print(X.shape)
print(y.shape)

(41860, 5)
(41860,)


### filtering the data: using perception condition only

In [4]:
# Convert labels to strings
y_str = y.astype(str)

# Create a mask to filter out the labels ending with '1'
mask = np.array([label.endswith('1') for label in y_str])

# Apply the mask to filter X and y
X_filtered = X[mask]
y_filtered = y[mask]

In [5]:
X_filtered.shape, y_filtered.shape

((10800, 5), (10800,))

### convert to mode labels

In [6]:
major_ids = [2, 3, 4, 12, 13, 14, 21, 23, 24]
minor_ids = [1, 11, 22]
mode_labels = []
for l in y_filtered:
    if int(str(l)[:-1]) in minor_ids:
        mode_labels.append(0)
    elif int(str(l)[:-1]) in major_ids:
        mode_labels.append(1)
### 0 = minor, 1 = major
#print(mode_labels[:40])
mode_labels = np.array(mode_labels)
print(mode_labels.shape)

(10800,)


### split test set away

In [7]:
X_filtered.shape

(10800, 5)

In [8]:
#leaving out participant 14 so it is not seen during training the svm
#1200 = 20 time windows * 60 trials for one participant
#15 trials for perception data 20*15 = 300
#45 trials for imagination data 20*45 = 900
X_train = X_filtered[:-1200, :] #X[:-1200, :] #X[:, :-60] #all but participant 14
X_test = X_filtered[-1200:, :] #X[-1200:, :] #X[:, -60:] #Participant 14 (last participant chosen as test set -> doing that for everything)
y_train = mode_labels[:-1200] #mode_labels[:-1200]
y_test = mode_labels[-1200:] #mode_labels[-1200:]

# Print the original distribution of the training set
print(f'Original class distribution in training set: {Counter(y_train)}')
print(f'Original class distribution in test set: {Counter(y_test)}')

Original class distribution in training set: Counter({1: 7200, 0: 2400})
Original class distribution in test set: Counter({1: 900, 0: 300})


### SMOTE

In [9]:
# Apply SMOTE to oversample the minority class
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Print the new distribution of the training set
print(f'Resampled class distribution: {Counter(y_train_resampled)}')

Resampled class distribution: Counter({1: 7200, 0: 7200})


# without majority voting

### Grid Search

In [9]:
param_grid = {
    "C" : [0.0001],
    "kernel" : ["poly"]
}

In [17]:
# Initialize the SVM classifier
svm_model = svm.SVC(class_weight='balanced', probability=True)

# Set up the grid search with cross-validation
grid_search = GridSearchCV(svm_model, param_grid, cv=StratifiedKFold(8), scoring='accuracy', verbose=1, n_jobs=-1)

# Fit the grid search to the training data
grid_search.fit(X_train_resampled, y_train_resampled)

# Print the best parameters and the best score
print(f'Best parameters: {grid_search.best_params_}')
print(f'Best cross-validation accuracy: {grid_search.best_score_}')

# Evaluate the best model on the test set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print(f'Test accuracy: {accuracy_score(y_test, y_pred)}')
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))
print('Classification Report:')
print(classification_report(y_test, y_pred))

print(len(y_pred), len(y_test))

Fitting 8 folds for each of 1 candidates, totalling 8 fits
Best parameters: {'C': 0.0001, 'kernel': 'poly'}
Best cross-validation accuracy: 0.4946527777777778
Test accuracy: 0.33666666666666667
Confusion Matrix:
[[233  67]
 [729 171]]
Classification Report:
              precision    recall  f1-score   support

           0       0.24      0.78      0.37       300
           1       0.72      0.19      0.30       900

    accuracy                           0.34      1200
   macro avg       0.48      0.48      0.33      1200
weighted avg       0.60      0.34      0.32      1200

1200 1200


# with majority voting on test set only

In [29]:
y_test_trials = [y_test[i] for i in range(0, len(y_test), 20)]
print(y_test_trials)
print(len(y_test_trials))
print(Counter(y_pred))

[1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1]
60
Counter({0: 962, 1: 238})


In [32]:
# Convert predictions to a numpy array for easier manipulation
predictions = np.array(y_pred)

# Initialize an empty list to store the majority voted predictions
majority_voted_predictions = []

# Loop through the predictions in chunks of 20
for i in range(0, len(predictions), 20):
    chunk = predictions[i:i+20]
    majority_vote = np.bincount(chunk).argmax()
    majority_voted_predictions.append(majority_vote)

# Ensure we have 60 majority voted predictions
assert len(majority_voted_predictions) == 60, "The result must contain exactly 60 majority voted predictions."

print(majority_voted_predictions)
print(Counter(majority_voted_predictions))

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Counter({0: 56, 1: 4})


In [34]:
print(classification_report(y_test_trials, majority_voted_predictions))

              precision    recall  f1-score   support

           0       0.23      0.87      0.37        15
           1       0.50      0.04      0.08        45

    accuracy                           0.25        60
   macro avg       0.37      0.46      0.22        60
weighted avg       0.43      0.25      0.15        60



### Grid Search and 8 fold cross validation for Random Forest Classifier

In [10]:
param_grid = {
    'n_estimators': [100, 250, 500],
    'min_samples_split': [2, 5, 10, 20],
    'max_features': [None, 'sqrt', 'log2'],
}


# Initialize the SVM classifier
rf_model = RandomForestClassifier()

# Set up the grid search with cross-validation
grid_search = GridSearchCV(rf_model, param_grid, cv=KFold(8), scoring='accuracy', verbose=1, n_jobs=-1)

# Fit the grid search to the training data
grid_search.fit(X_train_resampled, y_train_resampled)

# Print the best parameters and the best score
print(f'Best parameters: {grid_search.best_params_}')
print(f'Best cross-validation accuracy: {grid_search.best_score_}')

# Evaluate the best model on the test set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print(f'Test accuracy: {accuracy_score(y_test, y_pred)}')
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))
print('Classification Report:')
print(classification_report(y_test, y_pred))

Fitting 8 folds for each of 36 candidates, totalling 288 fits


Best parameters: {'max_features': None, 'min_samples_split': 2, 'n_estimators': 500}
Best cross-validation accuracy: 0.6577083333333333
Test accuracy: 0.6558333333333334
Confusion Matrix:
[[ 49 251]
 [162 738]]
Classification Report:
              precision    recall  f1-score   support

           0       0.23      0.16      0.19       300
           1       0.75      0.82      0.78       900

    accuracy                           0.66      1200
   macro avg       0.49      0.49      0.49      1200
weighted avg       0.62      0.66      0.63      1200



In [13]:
y_test_trials = [y_test[i] for i in range(0, len(y_test), 20)]
print(y_test_trials)
print(len(y_test_trials))
print(Counter(y_pred))

[1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1]
60
Counter({1: 989, 0: 211})


In [15]:
# Initialize an empty list to store the majority voted predictions
majority_voted_predictions = []

# Loop through the predictions in chunks of 20
for i in range(0, len(y_pred), 20):
    chunk = y_pred[i:i+20]
    majority_vote = np.bincount(chunk).argmax()
    majority_voted_predictions.append(majority_vote)

# Ensure we have 60 majority voted predictions
assert len(majority_voted_predictions) == 60, "The result must contain exactly 60 majority voted predictions."

print(majority_voted_predictions)
print(Counter(majority_voted_predictions))

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Counter({1: 60})


In [16]:
print(classification_report(y_test_trials, majority_voted_predictions))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00        15
           1       0.75      1.00      0.86        45

    accuracy                           0.75        60
   macro avg       0.38      0.50      0.43        60
weighted avg       0.56      0.75      0.64        60



/home/ucloud/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/ucloud/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/ucloud/.local/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1517: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
